## 08B Evaluation: Additional Analysis

#### Purpose
08a evaluated the models as classifiers. This notebook checks what a confusion matrix cannot show, and what the business needs:
- **The probabilities themselves:** log loss, Brier score, calibration and expected vs actual counts. The models are used for expected claims, expected lapses and pricing, so a 2% must mean 2%.
- **Where the models hold and where they fail:** by segment (state, flood zone, storm during the term), by year, and for lapse, over the life of a policy (**cohort survival analysis**).
- **How we would monitor them in production:** drift and alert rules.

It reads the predictions saved by **08a**, so the evaluation split is scored only once.

#### Steps:
0. Settings, and the predictions from 08a
1. **Claim model:** probability check, calibration, calibration by segment, stability over time, simulated monitoring
2. **Lapse model:** the same, plus the cohort survival analysis (retention of new business)
3. Appendix: bootstrap of the gains, policy history and linkage
4. Conclusions and save

#### Authors
Organized and prepared by Sebastian Carino, Ma. Loisa Celorico, Cyrill Agustin, John Navarro, and Justino Rotairo III.

- **Date Created:** 2026-11-07
- **Date Modified:** 2026-11-17

### Section 0. Load the libraries and working directory
Use the same setup cell as notebook 02 (repo root, data folders, reports folder). The shared evaluation code is in `src/evaluation.py`.

#### 0.1 Imports and paths

In [ ]:
%load_ext autoreload
%autoreload 2

# Find the repo root from any folder, so `src` can be imported
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from matplotlib.ticker import PercentFormatter

from src.paths import DOCS, PROCESSED, REPORTS, ensure_dirs
from src.db import connect
from src import evaluation as ev
pd.options.display.float_format = "{:,.4g}".format     # 4 significant digits, so small rates stay readable

ensure_dirs()

#### 0.2 Settings and the holdout switch
`EVAL_SPLIT` decides which out-of-time period is evaluated:

| `EVAL_SPLIT` | Years evaluated | When |
|---|---|---|
| `"test"` | OOT1, 2021–2022 | Dry run: as often as we like, while 07a/07b are being built |
| `"valid"` | OOT2, 2023 – Aug 2024 | **Once**, after 07a and 07b are final |

**08a and 08b must use the same value**: 08a scores the split and saves the predictions, 08b reads them (and stops if they were made for another split). In the dry run, nothing after 2022 is read. Every saved file carries the name of the split (`eval_test_…` or `eval_valid_…`), so the dry run never overwrites the final results.

We also rebuild the model registry and the naive benchmarks: the stability, cohort and monitoring steps score samples of other years.

In [ ]:
# Step 0.2 Settings, the predictions saved by 08a, the model registry and the naive benchmarks

EVAL_SPLIT = "test"        # change to "valid" ONCE, in 08a and 08b, after notebooks 07a and 07b are final
assert EVAL_SPLIT in ("test", "valid")
EVAL_NAME = ev.EVAL_NAMES[EVAL_SPLIT]
LAST_YEAR = 2022 if EVAL_SPLIT == "test" else 2024     # later years stay closed in the dry run
OUT = f"eval_{EVAL_SPLIT}"                             # prefix of every saved file
SEED = 42
MODEL_DIR, PRED_DIR = PROCESSED / "models", PROCESSED / "predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)
KINDS = pd.read_csv(DOCS / "feature_plan.csv").set_index("feature")["kind"].to_dict()   # num / cat / flag per feature

con = connect(db_path=None, memory_limit="6GB")
con.execute(f"CREATE OR REPLACE VIEW feats AS SELECT * FROM read_parquet('{PROCESSED / 'features.parquet'}')")

# The primary model of each target (shown in the charts): None = the advanced model with the lowest test log loss
# in its run record, otherwise the baseline. Set a model name, e.g. {"lapse": "lapse_rf"}, to choose it yourself.
PRIMARY_CHOICE = {"claim": None, "lapse": None}

# The models of each target, the primary model, and the two naive benchmarks rebuilt from the training split
registry, PRIMARY, base_rate, COLS = {}, {}, {}, {}
for kind in ev.TARGETS:
    registry[kind] = ev.load_registry(kind, MODEL_DIR, REPORTS)
    PRIMARY[kind] = ev.primary_model(registry[kind], PRIMARY_CHOICE.get(kind))
    base_rate[kind] = ev.build_benchmarks(con, kind)
    COLS[kind] = ev.model_columns(registry[kind])      # probability column of each model, in table order

N_BOOT = 200               # bootstrap samples (Step 3.1)
STABILITY_ROWS = 200_000   # rows per year (Steps 1.4, 2.4)
REFERENCE_ROWS = 100_000   # training rows per calendar quarter, the monitoring reference (Steps 1.5, 2.7)
QUARTER_ROWS = 100_000     # rows per quarter (Steps 1.5, 2.7)
COHORT_KEYS = 30_000       # new-business policies followed per cohort (Step 2.5)

run_08a = json.loads((REPORTS / f"{OUT}_08a_run.json").read_text())
assert run_08a["eval_split"] == EVAL_SPLIT, "08a scored another split: run 08a with the same EVAL_SPLIT first"
pred = {kind: pd.read_parquet(PRED_DIR / f"{OUT}_{kind}_predictions.parquet") for kind in ev.TARGETS}
for kind, p in pred.items():
    missing = set(COLS[kind].values()) - set(p.columns)
    assert not missing, f"{kind}: 08a has no predictions for {missing}: re-run 08a"
    print(f"{kind}: {len(p):,} rows of {EVAL_NAME} from 08a (scored {run_08a['scored_at']}); primary model {PRIMARY[kind]}")
QUARTERS = [(y, q) for y in range(2021, LAST_YEAR + 1) for q in range(1, 5) if not (y == 2024 and q == 4)]

### Section 1. Claim model

#### 1.1 Probability check
For every model and both benchmarks, overall and for each year. Let $y_i \in \{0,1\}$ be the outcome and $p_i$ the predicted probability.

$$\text{LogLoss} = -\frac{1}{n}\sum_{i=1}^{n}\Big[y_i\log p_i + (1-y_i)\log(1-p_i)\Big] \qquad \text{Brier} = \frac{1}{n}\sum_{i=1}^{n}(p_i - y_i)^2$$

$$\text{Skill} = 1 - \frac{\text{score}_{\text{model}}}{\text{score}_{\text{base rate}}} \qquad \text{A/E} = \frac{\sum_i y_i}{\sum_i p_i}$$

- **Log loss** and **Brier score**: lower is better; both are proper scoring rules (a model gets its best score only by giving the true probabilities). Log loss is also what we tuned on in 06a/06b.
- **Skill**: the improvement over giving everyone the training rate, on the same rows (0% = no better than the base rate).
- **A/E** (actual ÷ expected): 1 = the model expects as many claims as happened; above 1 = it under-predicts.

In [ ]:
# Step 1.1 Claim: log loss, Brier, skill and A/E, overall and per year
p = pred["claim"]
prob_claim = ev.probability_table([(EVAL_NAME, p), *[(str(y), g) for y, g in p.groupby("period_year")]], COLS["claim"])
prob_claim.style.format({"rows": "{:,}", "events": "{:,}", "actual_rate": "{:.3%}", "predicted_rate": "{:.3%}",
                          "a_to_e": "{:.3f}", "log_loss": "{:.5f}", "brier": "{:.6f}", "log_loss_skill": "{:+.2%}",
                          "brier_skill": "{:+.2%}"})

#### 1.2 Calibration
We sort the rows by each model's predicted probability, cut them into 10 equal groups (deciles) and compare the mean prediction $\bar p_d$ with the actual rate $\bar y_d$ in each.

- **Expected calibration error:** $\text{ECE} = \sum_{d}\frac{n_d}{n}\,|\bar{y}_d - \bar{p}_d|$, also shown as a share of the actual rate.
- **Calibration intercept and slope:** a logistic regression of the outcome on the model's own log-odds, $\text{logit}\,P(y_i=1) = a + b\,\text{logit}(p_i)$. $b = 1, a = 0$: perfect; $b < 1$: probabilities **too extreme** (overfitting); $b > 1$: **too timid**; $a \neq 0$ with $b \approx 1$: the overall level is off.
- **Expected vs actual:** the sum of the probabilities is the number of claims the model expects, $E[\text{claims}] = \sum_i p_i$, the most direct business check.

We do not use the Hosmer–Lemeshow test: with millions of rows it rejects even a tiny, harmless miscalibration.

In [ ]:
# Step 1.2 Claim: calibration by decile, ECE, intercept and slope, and expected vs actual per year
calib_claim, deciles_claim = ev.calibration_summary(pred["claim"], COLS["claim"], seed=SEED)
fig, ax = plt.subplots(figsize=(6, 5.4))
ev.plot_calibration(ax, deciles_claim, f"Claim: calibration by decile, {EVAL_NAME}", "claim")
plt.tight_layout()
plt.savefig(REPORTS / f"fig_{OUT}_claim_calibration.png", dpi=150)
plt.show()

p = pred["claim"]
expected_claim = p.groupby("period_year").agg(actual=("y", "sum"), **{f"expected: {n}": (c, "sum") for n, c in COLS["claim"].items()})
expected_claim.loc["all"] = expected_claim.sum()
display(expected_claim.style.format("{:,.0f}"))
display(deciles_claim[PRIMARY["claim"]].sort_index(ascending=False).style.format(
    {"rows": "{:,}", "events": "{:,}", "predicted": "{:.3%}", "actual": "{:.3%}", "actual_to_predicted": "{:.2f}"}))
calib_claim.style.format({"a_to_e": "{:.3f}", "ece": "{:.5f}", "ece_pct_of_rate": "{:.1%}", "intercept_a": "{:+.3f}", "slope_b": "{:.3f}"})

#### 1.3 Calibration by segment
A model can be right on average and wrong for whole groups. For each segment we show the actual rate, the **A/E** of each model and the log-loss skill. The chart shows the A/E of the primary model with a 95% interval from the number of claims (Poisson approximation): $\text{A/E} \times \big(1 \pm 1.96/\sqrt{\text{actual claims}}\big)$; the shaded band is 0.8–1.2, the "watch" band of the monitoring rules.

**Storm during the term** splits the rows by the strongest IBTrACS wind at the property during the term (an outcome-period measure, used only here). We expect the claim model to **under-predict where a storm hit and over-predict where none did**: no model known at the start of a term can know where the next storm will land. What matters is that the two balance out over the portfolio.

In [ ]:
# Step 1.3 (setup) The segments used for both targets
def top_states(frame, n=10):
    keep = frame["state"].value_counts().index[:n]
    return frame["state"].where(frame["state"].isin(keep), "other states")

SEGMENTS = {"state (largest 10)": top_states, "flood zone group": lambda f: f["zone_group"],
            "new business": lambda f: f["new_business"].map({1: "new business", 0: "renewal"}),
            "mandatory purchase": lambda f: f["mandatory_purchase"].map({1: "mandatory", 0: "voluntary"}),
            "storm during the term": lambda f: f["storm_in_term"]}

In [ ]:
# Step 1.3 Claim: A/E and skill by segment, for every model; chart for the primary model
seg_claim = {label: ev.segment_table(pred["claim"], COLS["claim"], key(pred["claim"])) for label, key in SEGMENTS.items()}
fig, ax = plt.subplots(figsize=(7, 9))
ev.plot_ae_by_segment(ax, seg_claim, PRIMARY["claim"], f"Claim: actual / expected by segment\n{PRIMARY['claim']}, {EVAL_NAME}")
plt.tight_layout()
plt.savefig(REPORTS / f"fig_{OUT}_claim_segments.png", dpi=150)
plt.show()
for label, t in seg_claim.items():
    print(label)
    display(t[["segment", "model", "rows", "events", "actual_rate", "a_to_e", "log_loss_skill"]].style.format(
        {"rows": "{:,}", "events": "{:,}", "actual_rate": "{:.3%}", "a_to_e": "{:.3f}", "log_loss_skill": "{:+.2%}"}))

#### 1.4 Stability over time
We score a random sample of every year from 2014 with the frozen models and follow the A/E and the log-loss skill: **in-sample** (2014–2020, the years the models learned from), **OOT1** (2021–2022) and **OOT2** (2023 – Aug 2024, final run only).

Claims are driven by storms, so the claim A/E swings from year to year even for a good model: high in a big storm year, low in a calm one. What we look for is a **trend** (A/E drifting away from 1) or skill falling below 0 in the out-of-time years.

In [ ]:
# Step 1.4 Claim: A/E and log-loss skill per year, 2014 to LAST_YEAR, on a sample of each year
t0 = time.time()
stability_claim = ev.stability_by_year(con, "claim", registry["claim"], base_rate["claim"], KINDS,
                                        range(2014, LAST_YEAR + 1), STABILITY_ROWS, seed=SEED)
print(f"scored {stability_claim['year'].nunique()} years in {(time.time() - t0) / 60:.1f} min")
fig, axes = plt.subplots(2, 1, figsize=(9, 6.5), sharex=True)
ev.plot_by_year(axes, stability_claim, COLS["claim"], f"Claim: stability over time")
plt.tight_layout()
plt.savefig(REPORTS / f"fig_{OUT}_claim_stability.png", dpi=150)
plt.show()
stability_claim.pivot_table(index=["year", "period"], columns="model", values=["a_to_e", "log_loss_skill"]).round(3)

#### 1.5 Simulated monitoring
Treat each quarter after the development years as if it had just arrived in production. The **reference** is a sample of the **same calendar quarter** in 2014–2020, because several features are seasonal (start month, rain in the last 90 days, days since the last storm).

**Population stability index**, with $r_b$ the reference share and $c_b$ the current share in bin $b$ (10 quantile bins for numbers, the 15 largest categories plus "other", missing as its own bin):
$$\text{PSI} = \sum_{b}\big(c_b - r_b\big)\,\ln\frac{c_b}{r_b}$$

| Check | OK | Watch | Action |
|---|---|---|---|
| Feature PSI | < 0.10 | 0.10 – 0.25 | > 0.25: review the feature's data and definition |
| Score PSI (the predicted probability) | < 0.10 | 0.10 – 0.25 | > 0.25: the mix of risks has changed; review with A/E |
| A/E per quarter | 0.9 – 1.1 | 0.8 – 0.9 or 1.1 – 1.2 | outside 0.8 – 1.2 for **two quarters in a row**: recalibrate |
| Log-loss skill per quarter | > 0 | — | below 0 for **two quarters in a row**: retrain |

A large feature PSI is not an error by itself. **Risk Rating 2.0** (from October 2021) changed the premiums, the deductible options and the rate-type codes, so the price and deductible features cross 0.25 in every quarter after 2021; features that measure time (days since the last storm, years since purchase, storms per year) also drift as the years pass. Drift matters when the A/E moves too.

**Claims need a different reading.** A single storm quarter pushes the claim A/E far above 1 with nothing wrong in the model, so for claims we read the A/E over **four rolling quarters** and act only on a persistent trend.

In [ ]:
# Step 1.5 Claim: feature PSI, score PSI and A/E per quarter, and the alert rules
t0 = time.time()
monitor_claim, psi_claim = ev.monitor_quarters(con, "claim", registry["claim"], base_rate["claim"], KINDS, PRIMARY["claim"],
                                                 QUARTERS, REFERENCE_ROWS, QUARTER_ROWS, seed=SEED)
alerts_claim = ev.apply_alert_rules(monitor_claim, rolling_quarters=4)
print(f"{len(QUARTERS)} quarters in {(time.time() - t0) / 60:.1f} min")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5.5), gridspec_kw={"width_ratios": [1.3, 1], "wspace": 0.45})
im = ev.plot_psi_heatmap(ax1, psi_claim, f"Claim: feature PSI vs the same quarter of 2014-2020 (largest 15)")
fig.colorbar(im, ax=ax1, shrink=0.8, label="PSI (0.10 watch, 0.25 review)")
x = np.arange(len(alerts_claim))
ax2.axhspan(0.8, 1.2, color="#eef3f9", zorder=0); ax2.axhspan(0.9, 1.1, color="#dbe7f4", zorder=0)
ax2.axhline(1, color="#555555", linewidth=1)
ax2.plot(x, alerts_claim["a_to_e"], color=ev.GREY, linewidth=1.5, marker="o", markersize=5, label="A/E per quarter")
if 4:
    ax2.plot(x, alerts_claim["a_to_e_used"], color="#2a6fb0", linewidth=2, label="A/E, 4 rolling quarters (used)")
ax2.set_xticks(x, alerts_claim["quarter"], rotation=90, fontsize=8)
ax2.set(title=f"Claim: actual / expected per quarter ({PRIMARY['claim']})", ylabel="A/E")
ax2.legend(frameon=False, fontsize=8)
ev.style_axes(ax2, grid="y")
plt.savefig(REPORTS / f"fig_{OUT}_claim_monitoring.png", dpi=150, bbox_inches="tight")
plt.show()
alerts_claim[["quarter", "score_psi", "max_feature_psi", "features_over_0.25", "a_to_e", "a_to_e_used", "log_loss_skill",
               "status", "action"]].style.format({"score_psi": "{:.3f}", "max_feature_psi": "{:.3f}", "a_to_e": "{:.3f}",
                                                   "a_to_e_used": "{:.3f}", "log_loss_skill": "{:+.2%}"})

### Section 2. Lapse model

#### 2.1 Probability check
The same measures as Step 1.1.

In [ ]:
# Step 2.1 Lapse: log loss, Brier, skill and A/E, overall and per year
p = pred["lapse"]
prob_lapse = ev.probability_table([(EVAL_NAME, p), *[(str(y), g) for y, g in p.groupby("period_year")]], COLS["lapse"])
prob_lapse.style.format({"rows": "{:,}", "events": "{:,}", "actual_rate": "{:.3%}", "predicted_rate": "{:.3%}",
                          "a_to_e": "{:.3f}", "log_loss": "{:.5f}", "brier": "{:.6f}", "log_loss_skill": "{:+.2%}",
                          "brier_skill": "{:+.2%}"})

#### 2.2 Calibration
As in Step 1.2: deciles, ECE, intercept and slope, and expected vs actual lapses.

In [ ]:
# Step 2.2 Lapse: calibration by decile, ECE, intercept and slope, and expected vs actual per year
calib_lapse, deciles_lapse = ev.calibration_summary(pred["lapse"], COLS["lapse"], seed=SEED)
fig, ax = plt.subplots(figsize=(6, 5.4))
ev.plot_calibration(ax, deciles_lapse, f"Lapse: calibration by decile, {EVAL_NAME}", "lapse")
plt.tight_layout()
plt.savefig(REPORTS / f"fig_{OUT}_lapse_calibration.png", dpi=150)
plt.show()

p = pred["lapse"]
expected_lapse = p.groupby("period_year").agg(actual=("y", "sum"), **{f"expected: {n}": (c, "sum") for n, c in COLS["lapse"].items()})
expected_lapse.loc["all"] = expected_lapse.sum()
display(expected_lapse.style.format("{:,.0f}"))
display(deciles_lapse[PRIMARY["lapse"]].sort_index(ascending=False).style.format(
    {"rows": "{:,}", "events": "{:,}", "predicted": "{:.3%}", "actual": "{:.3%}", "actual_to_predicted": "{:.2f}"}))
calib_lapse.style.format({"a_to_e": "{:.3f}", "ece": "{:.5f}", "ece_pct_of_rate": "{:.1%}", "intercept_a": "{:+.3f}", "slope_b": "{:.3f}"})

#### 2.3 Calibration by segment
As in Step 1.3. For lapse, the storm segment shows whether policyholders hit by a storm during the term renewed more or less than the model expected.

In [ ]:
# Step 2.3 Lapse: A/E and skill by segment, for every model; chart for the primary model
seg_lapse = {label: ev.segment_table(pred["lapse"], COLS["lapse"], key(pred["lapse"])) for label, key in SEGMENTS.items()}
fig, ax = plt.subplots(figsize=(7, 9))
ev.plot_ae_by_segment(ax, seg_lapse, PRIMARY["lapse"], f"Lapse: actual / expected by segment\n{PRIMARY['lapse']}, {EVAL_NAME}")
plt.tight_layout()
plt.savefig(REPORTS / f"fig_{OUT}_lapse_segments.png", dpi=150)
plt.show()
for label, t in seg_lapse.items():
    print(label)
    display(t[["segment", "model", "rows", "events", "actual_rate", "a_to_e", "log_loss_skill"]].style.format(
        {"rows": "{:,}", "events": "{:,}", "actual_rate": "{:.3%}", "a_to_e": "{:.3f}", "log_loss_skill": "{:+.2%}"}))

#### 2.4 Stability over time
As in Step 1.4. Lapse moves slowly, so a trend in the A/E is easier to see than for claims.

In [ ]:
# Step 2.4 Lapse: A/E and log-loss skill per year, 2014 to LAST_YEAR, on a sample of each year
t0 = time.time()
stability_lapse = ev.stability_by_year(con, "lapse", registry["lapse"], base_rate["lapse"], KINDS,
                                        range(2014, LAST_YEAR + 1), STABILITY_ROWS, seed=SEED)
print(f"scored {stability_lapse['year'].nunique()} years in {(time.time() - t0) / 60:.1f} min")
fig, axes = plt.subplots(2, 1, figsize=(9, 6.5), sharex=True)
ev.plot_by_year(axes, stability_lapse, COLS["lapse"], f"Lapse: stability over time")
plt.tight_layout()
plt.savefig(REPORTS / f"fig_{OUT}_lapse_stability.png", dpi=150)
plt.show()
stability_lapse.pivot_table(index=["year", "period"], columns="model", values=["a_to_e", "log_loss_skill"]).round(3)

#### 2.5 Cohort survival analysis: retention of new business
Steps 2.1–2.4 judge the lapse model one policy-year at a time. An insurer also needs to know **how long a new policy stays**: of the policies sold in a year, how many are still in force after 1, 2, 3, … years? That is a **survival** (retention) curve, the insurance version of a cohort or flow-rate analysis.

**Cohort:** all policies that start as **new business** (tenure under one year, with no earlier policy-year on the same key) in the same calendar year. We follow a random sample of 30,000 policies per cohort through their later policy-years; year $k$ is the $k$-th policy-year since the policy was sold.

**Flow table (life table).** In each year $k$, policies at risk either lapse (flow out: a cancellation during the term or a non-renewal), renew into year $k+1$, or are **censored**: we stop seeing them without a lapse, because the data end or a later policy-year left the modeling population. A censored policy is not counted as a lapse.
$$h_k = \frac{\text{lapsed in year } k}{\text{at risk at the start of year } k} \qquad S(k) = \prod_{j=1}^{k}\big(1 - h_j\big)$$
$h_k$ is the **lapse hazard** (the flow rate out) and $S(k)$ the **survival**, the share of the cohort still in force after $k$ years (the Kaplan–Meier estimator for yearly steps).

**What the model predicts.** Our lapse model is a logistic regression on policy-years, which is a **discrete-time hazard model**: each prediction $p_{ik}$ is the probability that policy $i$ lapses in year $k$, given that it is still in force. Averaging over the same policies at risk gives the model's hazard and survival, directly comparable with the actual curve:
$$\hat h_k = \frac{1}{n_k}\sum_{i \,\in\, \text{at risk at } k} p_{ik} \qquad \hat S(k) = \prod_{j=1}^{k}\big(1 - \hat h_j\big)$$
A gap between $S(k)$ and $\hat S(k)$ that **grows with $k$** means the model's errors add up over the years, which matters for premium-income forecasts several years ahead.

**Limits:** only cohorts sold from 2014 onward (policies in force in 2014 were sold earlier: left truncation); later cohorts have shorter curves (a year is shown while at least 500 policies are at risk); only the first spell, up to the first lapse; nothing after 2022 is read in the dry run.

In [ ]:
# Step 2.5 New-business cohorts: actual vs predicted lapse hazard and survival, year by year
MIN_AT_RISK = 500
t0 = time.time()
cohort_years = ev.build_cohort_keys(con, LAST_YEAR, COHORT_KEYS, seed=SEED)
parts = [ev.score_rows(con, "lapse", registry["lapse"], base_rate["lapse"], KINDS,
                       f"f.period_year <= {LAST_YEAR} AND f.policy_key IN (SELECT policy_key FROM cohort_keys WHERE cohort = {c})")[0]
         for c in cohort_years]
extra = con.sql(f"""
    SELECT f.period_id, CAST(f.CANCEL_IN_TERM AS INTEGER) AS cancelled, CAST(f.NON_RENEWAL AS INTEGER) AS not_renewed
    FROM feats f JOIN cohort_keys k USING (policy_key)
    WHERE f.in_pop_lapse AND f.period_year <= {LAST_YEAR}""").df()
cohort_rows = (pd.concat(parts, ignore_index=True)
                 .merge(con.sql("SELECT * FROM cohort_keys").df(), on="policy_key")
                 .merge(extra, on="period_id", how="left"))
del parts

# Year since new business, and only the first spell (up to and including the first lapse)
cohort_rows = cohort_rows[cohort_rows["period_start"] >= cohort_rows["entry_start"]].sort_values(["policy_key", "period_start"])
cohort_rows["year_k"] = cohort_rows.groupby("policy_key").cumcount() + 1
cohort_rows = cohort_rows[cohort_rows.groupby("policy_key")["y"].cumsum() - cohort_rows["y"] == 0]
print(f"{cohort_rows['policy_key'].nunique():,} policies in {len(cohort_years)} cohorts, {len(cohort_rows):,} policy-years "
      f"({(time.time() - t0) / 60:.1f} min)")

CURVES = {"model": f"p_{PRIMARY['lapse']}", "segment rule": "p_segment"}
tables = {c: ev.life_table(g, CURVES, MIN_AT_RISK) for c, g in cohort_rows.groupby("cohort")}
tables["all cohorts"] = ev.life_table(cohort_rows, CURVES, MIN_AT_RISK)
cohort_tables = pd.concat(tables, names=["cohort"]).reset_index()
cohort_check = cohort_tables[cohort_tables["year_k"].isin([1, 2, 3, 5])].assign(
    gap_model_pp=lambda t: 100 * (t["survival_model"] - t["survival_actual"]),
    gap_segment_pp=lambda t: 100 * (t["survival_segment rule"] - t["survival_actual"]))[
    ["cohort", "year_k", "at_risk", "survival_actual", "survival_model", "survival_segment rule", "gap_model_pp", "gap_segment_pp"]]

# One panel per cohort: actual vs predicted survival
ncols = 3
nrows = int(np.ceil(len(cohort_years) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3.2 * nrows), sharex=True, sharey=True, squeeze=False)
for ax, c in zip(axes.ravel(), cohort_years):
    t = tables[c]
    ax.plot(t.index, t["survival_actual"], color="#333333", linewidth=2, marker="o", markersize=5, label="actual (Kaplan-Meier)")
    ax.plot(t.index, t["survival_model"], color="#2a6fb0", linewidth=2, linestyle="--", marker="s", markersize=4,
            label=f"model ({PRIMARY['lapse']})")
    ax.plot(t.index, t["survival_segment rule"], color=ev.GREY, linewidth=1.5, linestyle=":", label="segment rule")
    ax.set_title(f"{c} cohort ({t['at_risk'].iloc[0]:,} policies)", fontsize=10)
    ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
    ev.style_axes(ax)
for ax in axes.ravel()[len(cohort_years):]:
    ax.set_visible(False)
for ax in axes[-1]:
    ax.set_xlabel("years since new business")
for ax in axes[:, 0]:
    ax.set_ylabel("share still in force")
axes[0, 0].set_ylim(0, 1.02)
axes[0, 0].legend(frameon=False, fontsize=8, loc="lower left")
plt.tight_layout()
plt.savefig(REPORTS / f"fig_{OUT}_lapse_cohort_survival.png", dpi=150)
plt.show()

display(tables["all cohorts"][["at_risk", "lapsed", "cancelled", "not_renewed", "censored", "hazard_actual", "hazard_model",
                               "survival_actual", "survival_model", "survival_segment rule"]].style.format(
    {"at_risk": "{:,}", "lapsed": "{:,}", "cancelled": "{:,}", "not_renewed": "{:,}", "censored": "{:,}",
     "hazard_actual": "{:.2%}", "hazard_model": "{:.2%}", "survival_actual": "{:.1%}", "survival_model": "{:.1%}",
     "survival_segment rule": "{:.1%}"}))
cohort_check.style.format({"at_risk": "{:,}", "survival_actual": "{:.1%}", "survival_model": "{:.1%}",
                           "survival_segment rule": "{:.1%}", "gap_model_pp": "{:+.1f}", "gap_segment_pp": "{:+.1f}"})

#### 2.6 Retention by segment at sale
The same curves for groups defined **at the time of sale**, pooling all cohorts: mandatory vs voluntary purchase (a lender's requirement should keep policies in force), and the flood zone group. The solid line is the actual retention, the dashed line the model's.

In [ ]:
# Step 2.6 Actual vs predicted retention by segment at the time of sale (all cohorts pooled)
at_sale = cohort_rows.sort_values("year_k").groupby("policy_key")[["mandatory_purchase", "zone_group"]].first()
seg_rows = cohort_rows.drop(columns=["mandatory_purchase", "zone_group"]).join(at_sale, on="policy_key")
SEG_AT_SALE = {"mandatory purchase at sale": seg_rows["mandatory_purchase"].map({1: "mandatory", 0: "voluntary"}),
               "flood zone group at sale": seg_rows["zone_group"].astype("string").fillna("(missing)")}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
retention_tables = {}
for ax, (label, key) in zip(axes, SEG_AT_SALE.items()):
    for i, (g, part) in enumerate(seg_rows.groupby(key)):
        t = ev.life_table(part, CURVES, MIN_AT_RISK)
        if t.empty:
            continue
        retention_tables[(label, g)] = t
        color = ev.MODEL_COLORS[i % len(ev.MODEL_COLORS)]
        ax.plot(t.index, t["survival_actual"], color=color, linewidth=2, marker="o", markersize=4, label=f"{g}: actual")
        ax.plot(t.index, t["survival_model"], color=color, linewidth=1.5, linestyle="--", label=f"{g}: model")
    ax.set(title=label, xlabel="years since new business", ylim=(0, 1.02))
    ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
    ax.legend(frameon=False, fontsize=8, ncol=2)
    ev.style_axes(ax)
axes[0].set_ylabel("share still in force")
plt.tight_layout()
plt.savefig(REPORTS / f"fig_{OUT}_lapse_cohort_segments.png", dpi=150)
plt.show()
(pd.concat(retention_tables, names=["segment_type", "segment"]).reset_index()
   .query("year_k in [1, 2, 3, 5]")[["segment_type", "segment", "year_k", "at_risk", "survival_actual", "survival_model"]]
   .style.format({"at_risk": "{:,}", "survival_actual": "{:.1%}", "survival_model": "{:.1%}"}))

#### Our findings (Steps 2.5–2.6, from the dry run)

**2.5 Retention of new business**

1. **A new flood policy has a short life.** Pooling the 2014–2022 cohorts, **24% of new policies lapse in their first year**, and only **64% are still in force after two years, 55% after three and 43% after five**. The lapse hazard is highest in the first year (23.8%), falls to 15.6% in the second and settles at about 10–12% a year from the fourth year. The first renewal is the main point of loss.
2. **Almost every lapse is a non-renewal.** In the first year, 64,284 of the 64,293 lapses were non-renewals and only 672 policies were cancelled during the term (some policies have both flags). Retention work should therefore focus on the **renewal date**, not on cancellations during the term.
3. **The model follows retention closely in the development years.** For the pooled cohorts, the predicted survival is within **1 percentage point** of the actual survival in each of the first five years (76.2% vs 75.8% after year 1, 43.2% vs 44.2% after year 5). The errors do not add up over the years, which matters for premium-income forecasts several years ahead. The segment rule is close on the pooled curve too, but it misses by more for individual cohorts: for the 2015 cohort it predicted 77.0% survival after year 1 against 70.5% actual, while the model predicted 73.1%.
4. **The gap opens in the out-of-time years, and it follows the calendar, not the age of the policy.** The model **over-predicts lapse in 2022**:
   - the 2022 cohort kept **79.5%** of its policies in year 1, while the model expected **70.7%** (hazard 20.5% vs 29.3%);
   - the 2021 cohort, in its second year (2022), lapsed at **11.0%** against **17.4%** predicted.

   Both are calendar year 2022, the first full year of **Risk Rating 2.0**: premiums and price features changed (Step 2.7), and the model reads the higher prices as a higher lapse risk, but fewer policyholders left than in the training years. The 2019 cohort shows the opposite in 2020–2021 (years 2–3): more lapses than expected, so the model's survival is 4.6 points too high after year 3.

**2.6 Retention by segment at sale**

5. **Mandatory purchase keeps policies in force**, as expected: 53.7% of mandatory-purchase policies are still in force after five years against 40.8% of voluntary ones. The model gets the voluntary group right (within 1 point every year) but is **slightly too optimistic for mandatory purchase from year 3 onward** (56.6% vs 53.7% after year 5), probably because some owners pay off or refinance their mortgage and the requirement ends.
6. **The flood zone at sale shows the clearest bias.**
   - **A and V zones:** the model is **too pessimistic**. It expects 77.7% retention after year 1 in zone A, against 81.8% actual, and the gap stays at about 4–5 points for five years.
   - **Zone X** (outside the high-risk area): it is **too optimistic**, 74.4% predicted vs 71.9% actual after year 1, and the gap grows to 4 points after year 5.

   The model under-uses the zone. Owners in the high-risk zones keep their cover longer than the model's features suggest, and owners outside them drop it sooner. Zone D has too few policies (271) to read.

**Conclusion.** As a hazard model, the lapse baseline gives realistic multi-year retention curves for the portfolio in the development years. Its errors come from **segments** (flood zone, mandatory purchase) and from the **2022 price change**, not from the length of the forecast. Before using it for premium-income forecasts, we would (a) let the advanced model (07b) learn interactions between zone and tenure, and (b) recalibrate it on post-Risk Rating 2.0 data, and confirm on the 2023–2024 holdout whether the 2022 over-prediction persists.

#### 2.7 Simulated monitoring
As in Step 1.5, with the quarterly A/E rule (lapse moves slowly, so no rolling window).

In [ ]:
# Step 2.7 Lapse: feature PSI, score PSI and A/E per quarter, and the alert rules
t0 = time.time()
monitor_lapse, psi_lapse = ev.monitor_quarters(con, "lapse", registry["lapse"], base_rate["lapse"], KINDS, PRIMARY["lapse"],
                                                 QUARTERS, REFERENCE_ROWS, QUARTER_ROWS, seed=SEED)
alerts_lapse = ev.apply_alert_rules(monitor_lapse, rolling_quarters=None)
print(f"{len(QUARTERS)} quarters in {(time.time() - t0) / 60:.1f} min")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5.5), gridspec_kw={"width_ratios": [1.3, 1], "wspace": 0.45})
im = ev.plot_psi_heatmap(ax1, psi_lapse, f"Lapse: feature PSI vs the same quarter of 2014-2020 (largest 15)")
fig.colorbar(im, ax=ax1, shrink=0.8, label="PSI (0.10 watch, 0.25 review)")
x = np.arange(len(alerts_lapse))
ax2.axhspan(0.8, 1.2, color="#eef3f9", zorder=0); ax2.axhspan(0.9, 1.1, color="#dbe7f4", zorder=0)
ax2.axhline(1, color="#555555", linewidth=1)
ax2.plot(x, alerts_lapse["a_to_e"], color=ev.GREY, linewidth=1.5, marker="o", markersize=5, label="A/E per quarter")
if None:
    ax2.plot(x, alerts_lapse["a_to_e_used"], color="#2a6fb0", linewidth=2, label="A/E, 4 rolling quarters (used)")
ax2.set_xticks(x, alerts_lapse["quarter"], rotation=90, fontsize=8)
ax2.set(title=f"Lapse: actual / expected per quarter ({PRIMARY['lapse']})", ylabel="A/E")
ax2.legend(frameon=False, fontsize=8)
ev.style_axes(ax2, grid="y")
plt.savefig(REPORTS / f"fig_{OUT}_lapse_monitoring.png", dpi=150, bbox_inches="tight")
plt.show()
alerts_lapse[["quarter", "score_psi", "max_feature_psi", "features_over_0.25", "a_to_e", "a_to_e_used", "log_loss_skill",
               "status", "action"]].style.format({"score_psi": "{:.3f}", "max_feature_psi": "{:.3f}", "a_to_e": "{:.3f}",
                                                   "a_to_e_used": "{:.3f}", "log_loss_skill": "{:+.2%}"})

### Section 3. Appendix: extra checks
Two checks that do not change the results but tell us how much to trust them, for both targets.

#### 3.1 Bootstrap by policy: is the gain over the benchmarks real?
We put a 95% interval on the **difference** in log loss between the primary model and each other model, on the same rows:
$$\Delta = \overline{\ell}_{\text{other}} - \overline{\ell}_{\text{primary}}, \qquad \ell_i = -\big[y_i\log p_i + (1-y_i)\log(1-p_i)\big]$$
$\Delta > 0$ means the primary model is better. Policy-years of the same policy are not independent, so we **resample policies, not rows**: each bootstrap sample gives every policy a random weight $w_k \sim \text{Poisson}(1)$, and $\Delta^{*} = \sum_k w_k D_k / \sum_k w_k n_k$, where $D_k$ is the sum of the differences over policy $k$'s rows and $n_k$ its number of rows. The 2.5th and 97.5th percentiles of 200 samples give the interval; an interval that does not include 0 means the gain is real for this period.

In [ ]:
# Step 3.1 Paired bootstrap of the log-loss difference, resampling policies (Poisson bootstrap)
bootstrap = pd.concat({kind: ev.bootstrap_log_loss(pred[kind], COLS[kind], f"p_{PRIMARY[kind]}", n_boot=N_BOOT, seed=SEED)
                       for kind in ev.TARGETS}, names=["target"]).reset_index(level=0)
bootstrap.style.format({"policies": "{:,}", "difference": "{:+.6f}", "ci_low": "{:+.6f}", "ci_high": "{:+.6f}"})

#### 3.2 Sensitivity to policy history and linkage
FEMA's data have no policy number, so notebook 03 rebuilt each policy's history with a policy key. Key collisions (5.2% of policy-years) were already excluded in notebook 04. What remains uncertain is the history of the policies we kept, so we compare the models on three kinds of history:

| History | Meaning | Why it matters |
|---|---|---|
| Continuous renewal | the previous term ended right before this one | the most reliable link |
| Returned after a gap | the key came back after a lapse | the link across the gap is less certain |
| First term on the key | no earlier term found | new business, or a link that was missed |

If the A/E or the skill is much worse for the last two, part of the error comes from linkage rather than from the model. We also show how the claims of the evaluation years were matched to terms (notebook 03).

In [ ]:
# Step 3.2 Claim-match quality in the evaluation years, and the models by policy history
match = con.sql(f"""
    SELECT match_status, COUNT(*) AS claims
    FROM read_parquet('{PROCESSED / 'claim_links' / '*.parquet'}')
    WHERE loss_year BETWEEN {2021 if EVAL_SPLIT == 'test' else 2023} AND {LAST_YEAR}
    GROUP BY ALL ORDER BY claims DESC""").df()
match["share"] = match["claims"] / match["claims"].sum()
display(match.style.format({"claims": "{:,}", "share": "{:.2%}"}))
history = {kind: ev.segment_table(pred[kind], COLS[kind], pred[kind]["history"]) for kind in ev.TARGETS}
for kind, t in history.items():
    print(f"{kind}: by policy history")
    display(t[["segment", "model", "rows", "events", "actual_rate", "a_to_e", "log_loss", "log_loss_skill"]].style.format(
        {"rows": "{:,}", "events": "{:,}", "actual_rate": "{:.3%}", "a_to_e": "{:.3f}", "log_loss": "{:.5f}", "log_loss_skill": "{:+.2%}"}))

### Section 4. Conclusions and save

#### 4.1 Our findings
*To be written after the final run (`EVAL_SPLIT = "valid"`). For each target:*
1. **Probability check (Steps 1.1, 2.1):** does the primary model beat the base rate and the segment rule on log loss and Brier, overall and in each year? Is the gain real (Step 3.1)?
2. **Calibration (Steps 1.2, 2.2):** A/E, ECE and slope: can the probabilities be used as they are, or do they need recalibration?
3. **Segments and history (Steps 1.3, 2.3, 3.2):** where is the model too high or too low, and is any of it linkage rather than the model?
4. **Stability and monitoring (Steps 1.4–1.5, 2.4, 2.7):** do the models drift from the development years to the out-of-time years, and which alert rules would have fired?
5. **Cohort survival (Steps 2.5–2.6):** does the predicted retention follow the actual retention of each cohort?
6. **Recommendation:** which model to use for each target, and what to do before it goes into production (recalibration, a refit on 2014–2022, monitoring rules).

#### 4.2 Save the results
Every table goes to `reports/` with the prefix `eval_test_` (dry run) or `eval_valid_` (final run), next to the charts saved above.

In [ ]:
# Step 4.2 Save the tables of 08b
for kind in ev.TARGETS:
    g = globals()
    g[f"prob_{kind}"].to_csv(REPORTS / f"{OUT}_{kind}_probability.csv", index=False)
    g[f"calib_{kind}"].to_csv(REPORTS / f"{OUT}_{kind}_calibration_summary.csv", index=False)
    pd.concat(g[f"deciles_{kind}"], names=["model", "decile"]).to_csv(REPORTS / f"{OUT}_{kind}_calibration_deciles.csv")
    g[f"expected_{kind}"].to_csv(REPORTS / f"{OUT}_{kind}_expected_vs_actual.csv")
    pd.concat(g[f"seg_{kind}"], names=["segment_type"]).to_csv(REPORTS / f"{OUT}_{kind}_by_segment.csv")
    g[f"stability_{kind}"].to_csv(REPORTS / f"{OUT}_{kind}_stability_by_year.csv", index=False)
    g[f"alerts_{kind}"].to_csv(REPORTS / f"{OUT}_{kind}_monitoring.csv", index=False)
    g[f"psi_{kind}"].to_csv(REPORTS / f"{OUT}_{kind}_feature_psi.csv")
    history[kind].to_csv(REPORTS / f"{OUT}_{kind}_by_history.csv", index=False)
cohort_tables.to_csv(REPORTS / f"{OUT}_lapse_cohort_life_tables.csv", index=False)
cohort_check.to_csv(REPORTS / f"{OUT}_lapse_cohort_check.csv", index=False)
bootstrap.to_csv(REPORTS / f"{OUT}_bootstrap.csv", index=False)
print(f"saved {OUT}_* tables and charts to {REPORTS}")